# 1. Import dependencies

In [11]:
!python -m pip install --quiet -r ../requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

from churn_prediction.paths import PROCESSED_DATA_PATH

In [13]:
df = pd.read_csv(PROCESSED_DATA_PATH)

X = df.drop(
    columns=['customerID', 'Churn']
)
y = df['Churn']

# 2. Split train, validation và test

## Split train + validation với test

In [14]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=69,
    stratify=y,
)

## Split train với validation

In [15]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.2,
    random_state=67,
    stratify=y_dev,
)

## Kiểm tra

In [16]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(4507, 19)
(1127, 19)
(1409, 19)
Churn
No     0.734635
Yes    0.265365
Name: proportion, dtype: float64
Churn
No     0.734694
Yes    0.265306
Name: proportion, dtype: float64
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


# 3. Preprocessing

## Định nghĩa các cột

In [17]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
yes_no_features = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]
binary_features = [
    "SeniorCitizen",
    *yes_no_features
]
multiclass_features = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

## Kiểm tra

In [18]:
all_features = (
    numerical_features
    + binary_features
    + multiclass_features
)
assert len(all_features) == len(set(all_features))
assert set(all_features) == set(X.columns)

## Binary Transformer

In [19]:
binary_transformer = OrdinalEncoder(
    categories=[
        ["No", "Yes"]
        for _ in yes_no_features
    ]
)

## ColumnTransformer

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features,
        ),
        (
            "binary",
            binary_transformer,
            yes_no_features,
        ),
        (
            "senior",
            "passthrough",
            ["SeniorCitizen"],
        ),
        (
            "multi",
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            ),
            multiclass_features,
        )
    ],
    remainder="drop",
)

## Nhìn thử Output

In [24]:
preprocessor.set_output(
    transform="pandas"
)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

In [25]:
X_train_transformed = preprocessor.fit_transform(
    X_train
)

### Kiểm tra

In [26]:
print(X_train.shape)
print(X_train_transformed.shape)

(4507, 19)
(4507, 41)


## Lấy tên cột trong Transformer

In [28]:
X_train_transformed.head()

,num__tenure,num__MonthlyCharges,num__TotalCharges,binary__Partner,binary__Dependents,binary__PhoneService,binary__PaperlessBilling,senior__SeniorCitizen,multi__gender_Female,multi__gender_Male,...,multi__StreamingMovies_No,multi__StreamingMovies_No internet service,multi__StreamingMovies_Yes,multi__Contract_Month-to-month,multi__Contract_One year,multi__Contract_Two year,multi__PaymentMethod_Bank transfer (automatic),multi__PaymentMethod_Credit card (automatic),multi__PaymentMethod_Electronic check,multi__PaymentMethod_Mailed check
5381,-0.022491,0.468439,0.071530,1.0,0.0,1.0,1.0,0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3251,0.180489,1.386233,0.779054,0.0,0.0,1.0,1.0,1,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
3142,-1.280967,-1.479195,-0.996332,0.0,0.0,1.0,0.0,0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
493,1.154793,-0.020941,0.711261,1.0,0.0,1.0,0.0,1,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3853,0.545853,0.267414,0.423994,0.0,0.0,1.0,0.0,0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


### Check numerical

In [30]:
X_train_transformed[
    [
        "num__tenure",
        "num__MonthlyCharges",
        "num__TotalCharges"
    ]
].describe()

,num__tenure,num__MonthlyCharges,num__TotalCharges
count,4.507000e+03,4.507000e+03,4.507000e+03
mean,-5.044901e-17,1.142985e-16,9.301536e-17
std,1.000111e+00,1.000111e+00,1.000111e+00
min,-1.321563e+00,-1.530275e+00,-1.005039e+00
25%,-9.561993e-01,-9.856958e-01,-8.294118e-01
50%,-1.442793e-01,1.800835e-01,-3.907053e-01
75%,9.518127e-01,8.358858e-01,6.548150e-01
max,1.601349e+00,1.778396e+00,2.814396e+00


# 4. Pipline

## Tạo pipeline

In [31]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
            ),
        ),
    ]
)

## Fit với train

In [32]:
pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['No','Yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying

## Validation

### Thử với threshold mặc định $0.5$

In [33]:
y_val_pred = pipeline.predict(
    X_val
)

y_val_prob_yes = pipeline.predict_proba(
    X_val
)[:, 1]

### Tính metrics

In [34]:
val_accuracy = accuracy_score(
    y_val,
    y_val_pred,
)

val_precision = precision_score(
    y_val,
    y_val_pred,
    pos_label="Yes",
)

val_recall = recall_score(
    y_val,
    y_val_pred,
    pos_label="Yes",
)

val_f1 = f1_score(
    y_val,
    y_val_pred,
    pos_label="Yes",
)

val_roc_auc = roc_auc_score(
    y_val,
    y_val_prob_yes,
)

val_ap = average_precision_score(
    y_val,
    y_val_prob_yes,
    pos_label="Yes",
)

In [35]:
print(f"Accuracy:          {val_accuracy:.4f}")
print(f"Precision (Yes):   {val_precision:.4f}")
print(f"Recall (Yes):      {val_recall:.4f}")
print(f"F1 (Yes):          {val_f1:.4f}")
print(f"ROC-AUC:           {val_roc_auc:.4f}")
print(f"Average Precision: {val_ap:.4f}")

Accuracy:          0.8110
Precision (Yes):   0.6641
Recall (Yes):      0.5819
F1 (Yes):          0.6203
ROC-AUC:           0.8500
Average Precision: 0.6690


Giống hệt với làm thủ công

Vì vậy, `Manual preprocessing` $\approx$ `ColumnTransformer preprocessing`

### Với threshold $0.37$

In [37]:
selected_threshold = 0.37

In [38]:
y_val_prob_yes = pipeline.predict_proba(
    X_val
)[:, 1]

y_val_pred_threshold = np.where(
    y_val_prob_yes >= selected_threshold,
    "Yes",
    "No",
)

### Tính F1

In [40]:
print(
    f1_score(
        y_val,
        y_val_pred_threshold,
        pos_label="Yes",
    )
)

0.6511627906976745


Khớp với validation của [04_proper_evaluation](./04_proper_evaluation.ipynb)